# 14. Test a fixed target with matched distal-leg timing controls

Does the fitted predictor use time-aligned opposite-leg context, and can the evidence survive support and geometry checks? We will keep the answer fixed while studying a narrowly defined change to permitted predictor inputs.

## Run this notebook independently

From the repository root, install the shared environment and start Jupyter:

```bash
uv sync --project experiments/aaai
AAAI_ITERATION02_RUN_DIR=/absolute/path/to/experiments/aaai/artifacts/iteration02/PROTOCOL_HASH \
  uv run --project experiments/aaai jupyter lab experiments/aaai
```

Replace `PROTOCOL_HASH` with the exact completed run you intend to inspect. The setup displays its producer command and verifies its recorded identity before showing results. It never chooses the newest run automatically. Alternatively, put `AAAI_ITERATION02_RUN_DIR` in the repository root `.env`. Do not create another environment file. Python-dotenv calls the real functions `load_dotenv` and `find_dotenv`; the aliases below implement the requested `load_env(find_env())` spelling without printing credentials.

Python 3.12 and the notebook, plotting, data and model dependencies are declared in the existing `pyproject.toml`. Select its kernel. This notebook does not depend on cells from another notebook. A full experiment fits models through the recorded producer command; this tutorial loads those real fitted outputs, checks them and teaches how to interpret them. Missing required outputs stop execution instead of triggering synthetic examples.

A small development run has its own protocol identity. To inspect it deliberately, also set `AAAI_ITERATION02_ALLOW_DEVELOPMENT=1`. Its outputs remain labeled incomplete. Do not use that option to bypass a failed evidence verifier.

The experiment is exploratory because the available sources informed earlier work. A dataset folder label is not a diagnosis independently established here. The focal endpoint is normal versus Parkinsons, stroke, cerebral palsy and myopathic labels. The mandatory broad endpoint uses all non-normal folders. The word “normal” refers to the dataset annotation, not a clinical clearance.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv as load_env, find_dotenv as find_env

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ambient").is_dir() and (p / "data/gavd").is_dir())
os.chdir(ROOT)
load_env(find_env())
BASE = ROOT / "experiments/aaai"
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.iteration02_notebook_support import (
    EvidenceRun, package_versions, show_diagram, show_video, show_status,
    optional_evidence, joint_table, source_role_table, plot_counts,
    prediction_example, example_losses, plot_prediction, plot_adequacy,
    plot_timing, plot_source_scores, within_fold_metrics, plot_metric_table,
    fold_json_records, fold_tables, cohort_table, mask_bank_summary,
    measured_window, plot_paired_differences, timing_video_example, external_replication, plot_condition_confusions,
)

run = EvidenceRun.from_env()
display(run.describe().T.rename(columns={0: "value"}))
display(package_versions())
if run.mode not in {"full", "full_study"}:
    display(Markdown("**DEVELOPMENT RUN. This is incomplete evidence and cannot supply final manuscript results.**"))

## 1. Freeze evaluation banks before comparing training masks

E_U preserves the original 16 uniform draws with seed 17. Every U-, J- and A-trained model answers that same question. A versus U is the primary masking-policy contrast, with A versus J and J versus U explaining possible differences.

E_leg is a separate diagnostic. It balances two distal-leg sides and starts 0, 2, 4 and 6, twice, giving 16 masks. Each draw hides an eight-token distal core and uniformly fills to 64 with dedicated seed 117. Only valid core targets enter its score. Reference source identities stay fixed, but its relational reference is fitted for that bank. A missing required reference makes the diagnostic unavailable; an outer-test source cannot replace it.

In [ ]:
show_diagram("mask_comparison")
display(mask_bank_summary(run))
show_status(run, "bank_status.json")

## 2. Compare within a column, not native task against native task

U/U and A/E_leg change both training and the tested observation task. Subtracting their losses cannot isolate a training-mask effect. Within either bank, all models use identical target keys, support and loss weights. Report each bank's coverage separately rather than intersecting every diagnostic into one tiny cohort.

The fixed-target timing control retains its own uniform parent-grid protocol across training arms. It does not inherit whichever native training mask produced a favorable score.

In [ ]:
scores = run.table("source_scores.csv", columns=("source_id", "policy", "bank"))
display(scores.groupby(["bank", "policy"]).source_id.nunique().unstack("policy"))

## 3. Build the no-wrap timing challenge on a common clock

Take a 48-frame parent. Fix the target and own-leg observations to frames 8:40, using zero-based half-open indexing. The aligned opposite-leg interval is also 8:40. Earlier and later donors are 4:36 and 12:44. No sample wraps from the end to the start.

For a permitted distal joint, subtract its permitted donor hip to obtain a hip-relative offset. Attach that offset to the permitted opposite hip at the original time, using the original fixed scale and body reference. Do not recover an unavailable hip from the unmasked pose. The intervention changes distal-leg input timing and geometry; it does not move the complete pelvis trajectory or create a physically possible patient.

In [ ]:
show_diagram("timing_controls")

## 4. Keep visibility, target and normalizer identities auditable

The 48-frame parent has 12 time patches and 192 allowed tokens. Draw its uniform mask before constructing any offset. Donor values keep their original visibility flags. Both donor and attachment hips must be observed and permitted.

Within each aligned-versus-shifted pair, restrict the aligned input to the same permitted support as the shifted input. Keep target coordinates, own-leg input, scale, fitted models, gates and reference parameters fixed. Earlier and later effects are separate primary diagnostics. A symmetric average uses only sources eligible for both and is secondary.

In [ ]:
show_status(run, "timing/raw_ledger_verification.json")
show_status(run, "timing/independent_raw_pair_verification.json")
timing = run.table("timing_pairs.csv")
display(timing.head(12))
print("Recorded paired timing rows:", len(timing))

## 5. Look at the source interval before looking at its effect

An input edit cannot tell us which neurological mechanism produced the original movement. First inspect the actual walking interval, missing points and direction of travel. A camera turn or person-tracking change may explain an apparent timing sensitivity.

The preview is tied to a recorded paired source when one is available. If the timing support is empty, the notebook reports that absence and does not substitute another intervention or manufacture a curve.

In [ ]:
timing_video_example(run)

## 6. Interpret raw prediction loss before calibrated anomaly scores

The primary timing effect is the frozen both-leg prediction loss under shifted context minus its loss under aligned context. Positive values mean that alignment helped this predictor on the same target and support. Changes in R or S are secondary because calibration introduces another transformation.

Report projected bone-length, joint-angle, displacement, speed and missingness changes. A finite crop does not exactly preserve a motion distribution. Even with hip-relative attachment, the body configuration changes. The result is therefore sensitivity to a compound timing-and-geometry input challenge, not an isolated physiological phase effect or a causal disease mechanism.

In [ ]:
timing_figure = plot_timing(run)
if timing_figure is not None:
    display(timing_figure); plt.close("all")
show_status(run, "timing_summary.json")
drift_columns = [name for name in timing if any(term in name for term in ("drift", "displacement", "speed", "bone", "angle", "count"))]
display(timing[["source_id", "policy", "variant", "offset", *drift_columns]].head(20))
paired_adequacy = run.table("timing/paired_support_adequacy.csv")
display(paired_adequacy)
print("Passed sparse-input gates:", int(paired_adequacy.paired_support_adequacy_passed.sum()), "of", len(paired_adequacy))
if not paired_adequacy.paired_support_adequacy_passed.all():
    display(Markdown("**The sparse paired-support adequacy gate failed for at least one comparison. Its timing effect is an input-sensitivity result, not evidence of physiological coordination.**"))

## 7. Preserve a useful null result without overinterpreting it

A null effect is informative only if the aligned sparse input still predicts competently and leaves measurable headroom. With almost no permitted opposite-leg tokens, neither timing nor its absence is identified. Do not relax the masking rule after seeing that outcome.

Independent phase normalization of each leg could erase the timing asymmetry being studied. Full-target phase can also leak the hidden answer. The primary model therefore preserves raw clock time, amplitude and asymmetry. A future phase-based analysis requires its own support and leakage audit.

## Preserve the evidence trail

The run directory is the unit of reproducibility. Keep its input hashes, source roles, fitted models, score references, target predictions, source scores, verification record and failed-comparison ledger together. A successful notebook execution establishes that this tutorial can reproduce its displayed analysis. It does not establish a neurological cause or a clinically valid classifier.

The [iteration02 protocol](docs/iteration02/protocol.md), [results](docs/iteration02/results.md), [limitations](docs/iteration02/limitations.md), and [draft02 readiness review](docs/draft02/README.md) must agree with the same run. The earlier draft01 and notebooks 00–11 remain historical evidence. New figures are written as SVG and PDF masters with PNG previews and an input-hash sidecar under this run's `figures/notebooks` directory.

In [ ]:
record_path = run.record('14_matched_distal_leg_timing_controls')
print("Recorded the exact real artifacts read by this notebook:", record_path)
print("This notebook did not replace the experiment producer or alter its source roles.")